In [1]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
from datetime import datetime, timezone
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0610 14:48:58.177000 79545 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
file_path = Path("/Users/mnatali/Projects/sentiment_analysis/brightdata_across_social_media_analysis/brightdata_social_exports/quora_datacenters_posts.json")

with file_path.open("r", encoding="utf-8") as f:
    quora_posts = json.load(f)

In [3]:
all_post_ids = []

env_post_ids = []
infr_post_ids = []
housing_post_ids = []
econ_post_ids = []
life_qual_post_ids = []
aesth_post_ids = []
gov_post_ids = []
not_useful_post_ids = []

environmental_keywords = ["environment", "pollution", "emissions", "diesel", "fumes", "light pollution", "water", "drought", "water consumption",  "waste", "carbon", "climate"]
infrastructure_utilities_keywords = ["utilities", "utility", "electric", "power", "grid", "blackout", "substation", "reliable", "reliability", "capacity", "infrastructure", "generator", "water"]
housing_keywords = ["housing", "house", "home", "rent", "property", "real estate", "house price", "housing price", "housing cost", "land cost", "land price", "gentrification"]
economic_keywords = ["job", "work", "revenue", "tax", "money", "rich", "salary", "economy", "investment", "business"]
life_quality_keywords = ["resident", "fumes", "noise", "loud", "quiet", "vibration", "light pollution", "sound", "smell", "traffic"]
aesthetics_keywords = ["landscape", "visual impact",  "aesthetics", "looks", "architecture", "design", "facade", "industrial", "windows", "dark"]
government_keywords = ["zoning", "zone", "planning", "approval", "permit", "city council", "bill", "mayor", "governor", "government"]

matched_posts = 0

for q_post in quora_posts:
    post_id = q_post["post_id"]
    all_post_ids.append(post_id)
    text = (q_post["post_text"])

    matched = False

    if any(kw in text for kw in environmental_keywords):
        env_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in infrastructure_utilities_keywords):
        infr_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in housing_keywords):
        housing_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in economic_keywords):
        econ_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in life_quality_keywords):
        life_qual_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in aesthetics_keywords):
        aesth_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in government_keywords):
        gov_post_ids.append(post_id)
        matched = True
    if not matched:
        not_useful_post_ids.append(post_id)
    else:
        matched_posts += 1

print("Total posts:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))

posts_by_id = {post["post_id"]: post for post in quora_posts}

Total posts: 1300
Total posts with a theme: 1182
Found environmental posts: 284
Found infrastructure posts: 628
Found housing posts: 621
Found economic posts: 907
Found life quality posts: 478
Found aesthetic posts: 350
Found government posts: 325


In [4]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of answers", "shares", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of answers", "shares", "views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_upvotes = []
    post_num_answers = []
    post_shares = []
    post_views = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["post_text"])
        post_dates.append(post["post_date"])
        post_upvotes.append(post["upvotes"])
        post_num_answers.append(post["over_all_answers"])
        post_shares.append(post["shares"])
        post_views.append(post["views"])


    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["upvotes"] = post_upvotes
    df1["number of answers"] = post_num_answers
    df1["shares"] = post_shares
    df1["views"] = post_views

    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    if theme == env_post_ids:
        df1["environment"] = True
    if theme == infr_post_ids:
        df1["infrastructure"] = True
    if theme == housing_post_ids:
        df1["housing"] = True
    if theme == econ_post_ids:
        df1["economy"] = True
    if theme == life_qual_post_ids:
        df1["life quality"] = True
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
    if theme == gov_post_ids:
        df1["government"] = True
    posts = pd.concat([posts, df1], ignore_index=True)

# Convert to proper dtypes
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "upvotes": "int64",
    "number of answers": "int64",
    "shares": "int64",
    "views": "int64",
    "environment": "bool",
    "infrastructure": "bool",
    "housing": "bool",
    "economy": "bool",
    "life quality": "bool",
    "aesthetics": "bool",
    "government": "bool",
    "AWS": "bool",
    "Amazon": "bool",
    "Google": "bool",
    "Microsoft": "bool",
    "Azure": "bool",
    "Meta": "bool",
    "Oracle": "bool",
    "Equinix": "bool",
    "Digital Realty": "bool",
    "IBM": "bool",
    "Facebook": "bool",
    "Apple": "bool",
    "QTS": "bool",
    "Vantage": "bool",
    "CyrusOne": "bool",
    "CoreSite": "bool"
})

len(posts['ids'])

3593

In [5]:
def remove_links(text):
    # Regex pattern to find typical URLs (http/https followed by non-whitespace characters)
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    # Replace found URLs with an empty string
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

sample_string = "The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...  [https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#](https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#)  A group of residents from the [Save Bren Mar From Data Center](https://sites.google.com/view/savebrenmar/home) group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial properties.  [https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board](https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board)Fairfax County considers enhancing data center guidelines"
result = remove_links(sample_string)
print(result)

posts["text"] = posts["text"].apply(remove_links)

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [6]:
grouping_cols = ["ids", "text", "date"]
posts = posts.groupby(grouping_cols, as_index=False).max()
theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
len(posts['ids'])

1182

In [7]:
# company_facility_terms = ["AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"]

# for post in posts["text"]:
#     for company in company_facility_terms:
#         if company.lower() in post.lower():
#             posts.loc[posts["text"] == post, company] = True

# posts.loc[posts[company_facility_terms].any(axis=1), company_facility_terms].head()

In [8]:
company_terms = ["AWS","Amazon","Google","Microsoft","Azure","Meta","Oracle","Equinix","Digital Realty","IBM","Facebook","Apple","QTS","Vantage","CyrusOne","CoreSite"]

for c in company_terms:
    posts[c] = posts["text"].str.contains(rf'\b{re.escape(c)}\b', case=False, na=False)

posts_with_company = posts[posts[company_terms].any(axis=1)]
posts_with_company.head()

,ids,text,date,upvotes,number of answers,shares,views,environment,infrastructure,housing,...,Oracle,Equinix,Digital Realty,IBM,Facebook,Apple,QTS,Vantage,CyrusOne,CoreSite
1,QW5zd2VyQDA6MTA0OTM5NTk1,I don’t know who is the most clever person to ...,2018-10-22T05:07:17.756Z,518,51,14,58942,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,QW5zd2VyQDA6MTA1MDgyOTE4,"The original idea social networks, connecting ...",2018-10-23T06:07:33.317Z,1,6,0,1100,True,False,False,...,False,False,False,False,True,False,False,False,False,False
4,QW5zd2VyQDA6MTA2MjEzNzA4,"In general, they made good updates to all but ...",2018-10-31T14:26:19.515Z,1,2,0,250,False,False,False,...,False,False,False,False,False,True,False,False,False,False
11,QW5zd2VyQDA6MTA4MDM0MDUy,I think it would be Microsoft. Well…at least i...,2018-11-14T14:59:29.858Z,20,88,0,3514,True,True,False,...,False,False,False,False,False,False,False,False,False,False
14,QW5zd2VyQDA6MTA5Mjc5MDYx,I was recently the victim of a military romanc...,2018-11-23T12:09:46.758Z,18,43,1,14912,False,False,True,...,False,False,False,False,True,False,False,False,False,False


In [9]:
# Make sure your pipeline returns all class scores, not just the top label
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    padding=True,
    max_length=512,
    top_k=None
)

tokenizer = classifier.tokenizer

def chunk_text(text, max_tokens=450):
    """
    Split one post into chunks small enough for RoBERTa.
    450 leaves room for special tokens and avoids hitting the 512 limit.
    """
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) == 0:
        return [""]

    chunks = []

    for i in range(0, len(token_ids), max_tokens):
        chunk_ids = token_ids[i:i + max_tokens]
        chunk = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk)

    return chunks


def aggregate_chunk_results(chunk_results):
    """
    Average sentiment scores across chunks.
    """
    scores = {
        "negative": [],
        "neutral": [],
        "positive": []
    }

    for result in chunk_results:
        for item in result:
            label = item["label"].lower()
            scores[label].append(item["score"])

    avg_scores = {
        label: float(np.mean(values)) if values else 0.0
        for label, values in scores.items()
    }

    final_label = max(avg_scores, key=avg_scores.get)

    return {
        "label": final_label,
        "score": avg_scores[final_label],
        "scores": avg_scores,
        "n_chunks": len(chunk_results)
    }

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [10]:
texts= posts["text"].fillna("").astype(str).tolist()
all_post_results = []

for text in tqdm(texts):
    chunks = chunk_text(text)

    chunk_results = classifier(
        chunks,
        truncation=True,
        padding=True,
        max_length=512,
        batch_size=32
    )

    post_result = aggregate_chunk_results(chunk_results)
    all_post_results.append(post_result)

posts["roberta"] = all_post_results

100%|██████████| 1182/1182 [01:57<00:00, 10.02it/s]


For reference, doing sentiment analysis on 1182 posts took 1 minutes and 45 seconds (only 24.5 seconds with the beast)

In [11]:
for i, row in posts.head(5).iterrows():
    print(f"Post {i}:")
    print(row["roberta"])
    print()

Post 0:
{'label': 'neutral', 'score': 0.7027899026870728, 'scores': {'negative': 0.013788693584501743, 'neutral': 0.7027899026870728, 'positive': 0.2834213972091675}, 'n_chunks': 1}

Post 1:
{'label': 'positive', 'score': 0.794970840215683, 'scores': {'negative': 0.02127142366953194, 'neutral': 0.18375774286687374, 'positive': 0.794970840215683}, 'n_chunks': 2}

Post 2:
{'label': 'neutral', 'score': 0.6924882531166077, 'scores': {'negative': 0.1340579092502594, 'neutral': 0.6924882531166077, 'positive': 0.17345385253429413}, 'n_chunks': 1}

Post 3:
{'label': 'neutral', 'score': 0.5739765167236328, 'scores': {'negative': 0.29355229437351227, 'neutral': 0.5739765167236328, 'positive': 0.13247119635343552}, 'n_chunks': 3}

Post 4:
{'label': 'positive', 'score': 0.8591183722019196, 'scores': {'negative': 0.023658063262701035, 'neutral': 0.11722354963421822, 'positive': 0.8591183722019196}, 'n_chunks': 2}



In [12]:
posts["name"] = posts["roberta"].apply(lambda x: x["label"])
posts["label"] = posts["name"]

posts["degree"] = posts["roberta"].apply(lambda x: x["score"])
posts = posts.drop(["roberta", "name"], axis=1)
posts.head(15)

,ids,text,date,upvotes,number of answers,shares,views,environment,infrastructure,housing,...,Digital Realty,IBM,Facebook,Apple,QTS,Vantage,CyrusOne,CoreSite,label,degree
0,QW5zd2VyQDA6MTA0ODM2NDY1,Software-defined network (SDN) and a tradition...,2018-10-21T11:20:20.865Z,0,6,0,1317,False,False,True,...,False,False,False,False,False,False,False,False,neutral,0.702790
1,QW5zd2VyQDA6MTA0OTM5NTk1,I don’t know who is the most clever person to ...,2018-10-22T05:07:17.756Z,518,51,14,58942,False,False,True,...,False,False,False,False,False,False,False,False,positive,0.794971
2,QW5zd2VyQDA6MTA1MDgyOTE4,"The original idea social networks, connecting ...",2018-10-23T06:07:33.317Z,1,6,0,1100,True,False,False,...,False,False,True,False,False,False,False,False,neutral,0.692488
3,QW5zd2VyQDA6MTA1ODI1Mzg4,Pre-Sales and Bid-Management have immense pote...,2018-10-28T18:50:07.259Z,87,3,2,35079,True,True,True,...,False,False,False,False,False,False,False,False,neutral,0.573977
4,QW5zd2VyQDA6MTA2MjEzNzA4,"In general, they made good updates to all but ...",2018-10-31T14:26:19.515Z,1,2,0,250,False,False,False,...,False,False,False,True,False,False,False,False,positive,0.859118
5,QW5zd2VyQDA6MTA2MzYxOTMw,"That is a big subject, depending on the scale ...",2018-11-01T16:59:46.862Z,0,2,1,250,True,False,False,...,False,False,False,False,False,False,False,False,neutral,0.773825
6,QW5zd2VyQDA6MTA2MzYyMjMw,"Since about 2016, maybe 2015, I have been seei...",2018-11-01T17:02:28.168Z,0,3,0,209,False,True,False,...,False,False,False,False,False,False,False,False,neutral,0.743817
7,QW5zd2VyQDA6MTA2NzE5MzI=,"While it is possible to have similar climates,...",2015-03-20T17:00:49.216Z,2,2,0,370,True,False,False,...,False,False,False,False,False,False,False,False,neutral,0.843803
8,QW5zd2VyQDA6MTA3MzIyNzIz,"I’ll give you two choices (or options, if you ...",2018-11-09T03:12:22.372Z,0,8,0,6798,False,False,True,...,False,False,False,False,False,False,False,False,neutral,0.774406
9,QW5zd2VyQDA6MTA3NjY0ODc1,"A2A. All will benefit in small ways, but the b...",2018-11-11T22:26:52.306Z,9,114,0,771,False,True,True,...,False,False,False,False,False,False,False,False,neutral,0.387192


In [13]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_theme_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    pos = theme_posts.loc[theme_posts["label"] == "positive", "degree"].sum()
    neg = theme_posts.loc[theme_posts["label"] == "negative", "degree"].sum()
    total = theme_posts["degree"].sum()
    return (pos-neg)/total

print("Average sentiment of environmental posts:", avg_theme_sentiment_calculation("environment"))
print("Average sentiment of infrastructural posts:", avg_theme_sentiment_calculation("infrastructure"))
print("Average sentiment of housing-related posts:", avg_theme_sentiment_calculation("housing"))
print("Average sentiment of economic posts:", avg_theme_sentiment_calculation("economy"))
print("Average sentiment of life-quality-related posts:", avg_theme_sentiment_calculation("life quality"))
print("Average sentiment of aesthetics-related posts:", avg_theme_sentiment_calculation("aesthetics"))
print("Average sentiment of governmental posts:", avg_theme_sentiment_calculation("government"))

Average sentiment of environmental posts: 0.18173637312909444
Average sentiment of infrastructural posts: 0.2148360864875885
Average sentiment of housing-related posts: 0.12282057059131955
Average sentiment of economic posts: 0.20231825539159903
Average sentiment of life-quality-related posts: 0.2179816965903178
Average sentiment of aesthetics-related posts: 0.20426853310317308
Average sentiment of governmental posts: 0.18229894151468315


In [14]:
## total average sentiment calculation

pos = posts.loc[posts["label"] == "positive", "degree"].sum()
neg = posts.loc[posts["label"] == "negative", "degree"].sum()
total = posts["degree"].sum()
print("Average sentiment for all of quora: ", (pos-neg)/total)

Average sentiment for all of quora:  0.1656455204474702


In [15]:
pos = posts.loc[posts["label"] == "positive"]
neg = posts.loc[posts["label"] == "negative"]
neutral = posts.loc[posts["label"] == "neutral"]

print("Percent of posts that are neutral:", len(neutral)/len(posts))
print("Percent of posts that are negative:", len(neg)/len(posts))
print("Percent of posts that are positive:", len(pos)/len(posts))

really_pos = pos.loc[posts["degree"] > 0.70]
really_neg = neg.loc[posts["degree"] > 0.70]

if len(neg) != 0:
    print("Percent of negative posts that are really negative:", len(really_neg)/len(neg))
if len(pos) != 0:
    print("Percent of positive posts that are really positive:", len(really_pos)/len(pos))

Percent of posts that are neutral: 0.6692047377326565
Percent of posts that are negative: 0.08967851099830795
Percent of posts that are positive: 0.24111675126903553
Percent of negative posts that are really negative: 0.2169811320754717
Percent of positive posts that are really positive: 0.45964912280701753


In [16]:
def avg_sentiment_calculation(dataset):
    pos = dataset.loc[dataset["label"] == "positive", "degree"].sum()
    neg = dataset.loc[dataset["label"] == "negative", "degree"].sum()
    total = dataset["degree"].sum()
    if total != 0:
        return (pos-neg)/total
    else:
        return 0

# Extract year from the date string
posts['year'] = pd.to_datetime(posts['date']).dt.year

# Create datasets for each year (efficient approach using a dictionary)
year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

# Access them like:
posts_2010 = year_datasets[2010]
posts_2020 = year_datasets[2020]

for i in range(2010, 2027):
    print(f"Number of posts from {i}: ", len(year_datasets[i]))
    if(avg_sentiment_calculation(year_datasets[i]) != 0):
        print(f"Average sentiment for quora posts from {i}: ", avg_sentiment_calculation(year_datasets[i]))

Number of posts from 2010:  3
Average sentiment for quora posts from 2010:  0.2908418308849044
Number of posts from 2011:  16
Average sentiment for quora posts from 2011:  0.227141871773505
Number of posts from 2012:  12
Average sentiment for quora posts from 2012:  0.2132054058556707
Number of posts from 2013:  14
Average sentiment for quora posts from 2013:  0.05119890191455249
Number of posts from 2014:  25
Average sentiment for quora posts from 2014:  0.1127974206507187
Number of posts from 2015:  37
Average sentiment for quora posts from 2015:  0.0655517466889031
Number of posts from 2016:  71
Average sentiment for quora posts from 2016:  0.009255485160942298
Number of posts from 2017:  116
Average sentiment for quora posts from 2017:  0.11568048644775071
Number of posts from 2018:  138
Average sentiment for quora posts from 2018:  0.034929371210866526
Number of posts from 2019:  135
Average sentiment for quora posts from 2019:  0.06543869895303989
Number of posts from 2020:  101


In [17]:
def sentiment_by_company(company1, company2=None):
    company_posts = posts[
        (posts[company1] == True) | (posts[company2] == True) if company2 else (posts[company1] == True)
    ]
    return avg_sentiment_calculation(company_posts)

print("Number of Amazon-related posts:", len(posts[posts["AWS"] == True]) + len(posts[posts["Amazon"] == True]))
print("Average sentiment of AWS-related posts:", sentiment_by_company("AWS", "Amazon"))
print("Number of Google-related posts:", len(posts[posts["Google"] == True]))
print("Average sentiment of Google-related posts:", sentiment_by_company("Google"))
print("Number of Microsoft-related posts:", len(posts[posts["Microsoft"] == True]) + len(posts[posts["Azure"] == True]))
print("Average sentiment of Microsoft-related posts:", sentiment_by_company("Microsoft", "Azure"))
print("Number of Meta-related posts:", len(posts[posts["Meta"] == True]) + len(posts[posts["Facebook"] == True]))
print("Average sentiment of Meta-related posts:", sentiment_by_company("Meta"))
print("Number of Oracle-related posts:", len(posts[posts["Oracle"] == True]))
print("Average sentiment of Oracle-related posts:", sentiment_by_company("Oracle"))
print("Number of Equinix-related posts:", len(posts[posts["Equinix"] == True]))
print("Average sentiment of Equinix-related posts:", sentiment_by_company("Equinix"))
print("Number of Digital Realty-related posts:", len(posts[posts["Digital Realty"] == True]))
print("Average sentiment of Digital Realty-related posts:", sentiment_by_company("Digital Realty"))
print("Number of IBM-related posts:", len(posts[posts["IBM"] == True]))
print("Average sentiment of IBM-related posts:", sentiment_by_company("IBM"))
print("Number of Apple-related posts:", len(posts[posts["Apple"] == True]))
print("Average sentiment of Apple-related posts:", sentiment_by_company("Apple"))
print("Number of QTS-related posts:", len(posts[posts["QTS"] == True]))
print("Average sentiment of QTS-related posts:", sentiment_by_company("QTS"))
print("Number of Vantage-related posts:", len(posts[posts["Vantage"] == True]))
print("Average sentiment of Vantage-related posts:", sentiment_by_company("Vantage"))
print("Number of CyrusOne-related posts:", len(posts[posts["CyrusOne"] == True]))
print("Average sentiment of CyrusOne-related posts:", sentiment_by_company("CyrusOne"))
print("Number of CoreSite-related posts:", len(posts[posts["CoreSite"] == True]))
print("Average sentiment of CoreSite-related posts:", sentiment_by_company("CoreSite"))

Number of Amazon-related posts: 194
Average sentiment of AWS-related posts: 0.23497398283354745
Number of Google-related posts: 284
Average sentiment of Google-related posts: 0.23294115672020974
Number of Microsoft-related posts: 181
Average sentiment of Microsoft-related posts: 0.2606934369545891
Number of Meta-related posts: 112
Average sentiment of Meta-related posts: 0.3639404323959298
Number of Oracle-related posts: 15
Average sentiment of Oracle-related posts: 0.13680608722312654
Number of Equinix-related posts: 1
Average sentiment of Equinix-related posts: 1.0
Number of Digital Realty-related posts: 1
Average sentiment of Digital Realty-related posts: 0.0
Number of IBM-related posts: 48
Average sentiment of IBM-related posts: 0.21241231448173353
Number of Apple-related posts: 143
Average sentiment of Apple-related posts: 0.2050112609845713
Number of QTS-related posts: 0
Average sentiment of QTS-related posts: 0
Number of Vantage-related posts: 0
Average sentiment of Vantage-rela

In [18]:
def analysis_of_viral_posts(min):
    viral_posts = posts.loc[posts["views"] >= min]
    print("What's considered viral: posts with over", min, "views")
    print("Number of viral posts:", len(viral_posts))
    print("Average sentiment of viral posts:", avg_sentiment_calculation(viral_posts))
    print("Average sentiment of non-viral posts:", avg_sentiment_calculation(posts.loc[posts["views"] < min]))
    print("Amount percent of viral posts that are polar (degree) > 0.70: ", len(viral_posts.loc[viral_posts["degree"] > 0.70])/len(viral_posts))

analysis_of_viral_posts(50)
print("\n")
analysis_of_viral_posts(100)

What's considered viral: posts with over 50 views
Number of viral posts: 1102
Average sentiment of viral posts: 0.15895242961435535
Average sentiment of non-viral posts: 0.2540503159572255
Amount percent of viral posts that are polar (degree) > 0.70:  0.42105263157894735


What's considered viral: posts with over 100 views
Number of viral posts: 998
Average sentiment of viral posts: 0.15058020837260883
Average sentiment of non-viral posts: 0.2442256295781247
Amount percent of viral posts that are polar (degree) > 0.70:  0.4198396793587174
